# Tier 1 version 2: scale, shape and alignment over the alpha by weight-decay grid

## What this notebook found

The report's hypothesis H says that generalisation begins when the centred alignment crosses a threshold
that every cell shares, and that the grokking time does not depend on the kernel scale once the alignment
is known. The three parts of that claim come out differently, and this section states where each stands
before any figure is drawn.

**H0, the confound, is established, and has moved.** The usual movement statistic D does not identify the
mechanism. In the baseline cell 98 percent of D at the generalisation event is the kernel growing rather
than turning, and across the grid D varies by a factor of 13 at the event while the alignment varies by a
factor of 1.34. That material now sits beside the reproduction in `tier0_v2.ipynb`, because it is the one
contribution that does not depend on anything else holding.

**H1, a shared threshold, holds and is not an artefact.** The alignment at the generalisation event varies
by a factor of 1.34 across all 40 saved runs that generalised, and by 1.18 across the twelve seed 0 cells
of section 6. Both are inside the report's kill criterion of about two. Section 8 also asks
whether that tightness means anything, by reading every run at another run's generalisation step. The
spread widens from 6.8 percent to a median of 11.1 percent, so the alignment is genuinely locked to the
event rather than merely confined to a narrow band all through training.

**Three findings qualify H1, and all three belong in the report.** The alignment is at least as
characteristic of the memorisation event, where its spread is 5.6 percent, as of the generalisation event.
The shape term R is more strongly locked to the event than the alignment is, tightening by a factor of 3.4
against the alignment's 1.6, and R is blind to the task. And crossing the threshold is not sufficient: the
three alpha 2 runs at decay 0.0003 cross it and never generalise. Since H claims that the task-relevant
part of the shape change is what sets the timing, a task-blind term that tracks the event at least as well
is the result that bears on the hypothesis.

**H2, that scale adds nothing once alignment is known, already fails, and the failure is not stable.**
Version 1 of this notebook reports a partial correlation of the grokking time with the scale term, holding
the alignment-crossing time fixed, of -0.730 with an interval of -0.912 to -0.424. That excludes zero, so
the kill criterion the report sets for H2, a scale term that stays significant with the alignment in the
model, is already met on version 1's own numbers. Section 8 says so plainly and then asks how firm the
result is. Version 1 uses 22 of the 51 runs, dropping every censored run and every run whose alignment
peaked below the mean of the alignments it was averaged from, so it is not the censored regression the
report specifies. Running that regression, the scale term is significant alongside the alignment in seven
of twelve specifications and changes sign across the range of alignment levels. H2 fails as measured, but
the grid does not pin down by how much or in which direction.

**What this means for the contribution.** The decomposition as a timing variable stands on its own. The
alignment threshold is a real effect worth reporting, with its sufficiency failure stated. Two parts of H
are not supported on this grid and the report should say so: scale still predicts the timing once the
alignment is known, and the task-blind shape term tracks the generalisation event at least as well as the
task-referenced alignment does. The second is the one that bears on what makes H distinctive, and section
7 is the figure that makes it visible.

## Grid

Width 100 in NTK parameterisation at base rate 100, alpha in 0.5, 1 and 2, eta times kappa in 0, 0.00001,
0.00003, 0.0001 and 0.0003. The proposal's decay values of 0.001 and above shrink the weights within a few
thousand steps, so the axis was refined downwards. Runs are 100,000 steps and every saved seed is used
where seeds exist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ntk_lib as L  # training loop, crossings, kernel terms and plot helpers shared by the tier notebooks

L.set_plot_style()

ALPHAS = [0.5, 1.0, 2.0]
DECAYS = [0.0, 1e-5, 3e-5, 1e-4, 3e-4]
SEEDS = [0, 1, 2, 3, 4]
STEPS = 100000

# Test accuracy at which the generalisation event is declared; the memorisation event is always train accuracy 1.
# Change this one number to move every crossing, table and figure in the notebook.
TEST_ACC_LEVEL = 1.0

def saved(alpha, ek, seed):
    """True if this seed of the cell is already trained and saved under results/."""
    return (L.RESULTS / (L.cell_name(100, alpha, ek, seed) + ".json")).exists()

# The main grid is seed 0 of every cell, trained if missing. Extra seeds are gathered where saved for the seed checks.
grid = {(a, ek): L.load_cell(100, a, ek, 0, STEPS, eval_interval=500) for a in ALPHAS for ek in DECAYS}
rows = {key: L.summarise_cell(run, test_level=TEST_ACC_LEVEL) for key, run in grid.items()}
extra = {(a, ek, s): L.load_cell(100, a, ek, s, STEPS, eval_interval=500)
         for a in ALPHAS for ek in DECAYS for s in SEEDS[1:] if saved(a, ek, s)}
extra_rows = {key: L.summarise_cell(run, test_level=TEST_ACC_LEVEL) for key, run in extra.items()}


## 1. The grid

One row per cell at seed 0. D is the movement statistic at the generalisation event, recovered from S and R through Equation 1 of the proposal. Kernel terms are blank when that event is censored.


In [ ]:
# One line per cell: the two events, the grokking time, and the kernel terms read at the generalisation event.
print(f"{'alpha':>5} {'eta kappa':>9} {'t_train':>8} {'t_test':>8} {'t_grok':>8} {'A@test':>7} {'A peak':>7} {'S@test':>7} {'R@test':>7} {'D@test':>7}")
for (a, ek), r in rows.items():
    t_grok = L.fmt(r["t_grok"]) + ("+" if r["censored"] and r["t_grok"] is not None else "")  # plus marks a lower bound
    print(f"{a:>5g} {ek:>9g} {L.fmt(r['t_train']):>8} {L.fmt(r['t_test']):>8} {t_grok:>8} {L.fmt(r['A_at_test']):>7} "
          f"{L.fmt(r['A_peak']):>7} {L.fmt(r['S_at_test'], 2):>7} {L.fmt(r['R_at_test']):>7} {L.fmt(r['D_at_test'], 2):>7}")


## 2. Each cell as a path through scale, shape and alignment

The three centred kernel terms as axes. Every cell starts at zero scale and zero shape change with an
alignment near 0.09. Decay bends the paths towards smaller scale without lowering the alignment reached,
and the generalisation dots of a panel cluster in shape and alignment while spreading in scale.

The demonstration that two cells can share a movement statistic and differ in mechanism has moved to
`tier0_v2.ipynb`, sections 5 to 8.

In [ ]:
# Three 3D scenes, one per alpha; each decay level is one path through (S, R, A) plus one dot at its generalisation event.
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scene"}] * 3], subplot_titles=[f"alpha {a:g}" for a in ALPHAS],
                    horizontal_spacing=0.02)
decay_colours = dict(zip(DECAYS, L.RAMP))
for col, a in enumerate(ALPHAS, start=1):
    for ek in DECAYS:
        run = grid[(a, ek)]
        h = run["history"]
        fig.add_trace(go.Scatter3d(x=h["S_c"], y=h["R_c"], z=h["A_t"], mode="lines", line=dict(color=decay_colours[ek], width=4),
                                   name=f"eta kappa {ek:g}", legendgroup=str(ek), showlegend=(col == 1), customdata=h["step"],
                                   hovertemplate="step %{customdata}<br>S %{x:.2f}<br>R %{y:.3f}<br>A %{z:.4f}<extra>alpha " + f"{a:g}, eta kappa {ek:g}</extra>"),
                      row=1, col=col)
        t = rows[(a, ek)]["t_test"]
        if t is not None:
            fig.add_trace(go.Scatter3d(x=[L.value_at_step(run, "S_c", t)], y=[L.value_at_step(run, "R_c", t)], z=[L.value_at_step(run, "A_t", t)],
                                       mode="markers", marker=dict(color=decay_colours[ek], size=5, line=dict(color="white", width=1)),
                                       legendgroup=str(ek), showlegend=False,
                                       hovertemplate=f"generalisation event at step {t}<br>S %{{x:.2f}}<br>R %{{y:.3f}}<br>A %{{z:.4f}}<extra>alpha {a:g}, eta kappa {ek:g}</extra>"),
                          row=1, col=col)
scene = dict(xaxis_title="scale S_t", yaxis_title="rotation R_t", zaxis_title="alignment A_t",
             camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)))
fig.update_layout(scene=scene, scene2=scene, scene3=scene, height=520, width=1250, template="plotly_white",
                  title="Each cell as a path through scale, rotation and alignment; dots at the generalisation event",
                  legend_title_text="line colour", margin=dict(t=70, b=10, l=0, r=0))
fig.show()


## 3. The three pairs of terms, with a line through the generalisation events

The 3D paths above projected onto each pair of axes. Faint lines are the seed 0 paths of every cell and the points are the generalisation events of every saved seed, coloured by alpha with decay as shade. A dashed least-squares line is fitted through the events of each alpha, with its slope and correlation in the legend.


In [ ]:
pairs = [("S_c", "R_c", "scale S_t", "rotation R_t"), ("S_c", "A_t", "scale S_t", "alignment A_t"), ("R_c", "A_t", "rotation R_t", "alignment A_t")]
alpha_colours = {0.5: L.RAMP[0], 1.0: L.RAMP[2], 2.0: L.RAMP[4]}
shade = dict(zip(DECAYS, np.linspace(0.45, 1.0, len(DECAYS))))  # lighter for no decay, full for the strongest

# Every saved run with a generalisation event: its three terms at that event.
events = []
for (a, ek), r in rows.items():
    if r["t_test"] is not None:
        events.append((a, ek, 0, r["S_at_test"], r["R_at_test"], r["A_at_test"]))
for (a, ek, s), r in extra_rows.items():
    if r["t_test"] is not None:
        events.append((a, ek, s, r["S_at_test"], r["R_at_test"], r["A_at_test"]))
col = {"S_c": 3, "R_c": 4, "A_t": 5}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (kx, ky, lx, ly) in zip(axes, pairs):
    for (a, ek), run in grid.items():  # faint paths, seed 0
        h = run["history"]
        ax.plot(h[kx], h[ky], color=alpha_colours[a], lw=0.8, alpha=0.25)
    x = np.array([e[col[kx]] for e in events]); y = np.array([e[col[ky]] for e in events])
    for e, xi, yi in zip(events, x, y):
        ax.plot([xi], [yi], "o", color=alpha_colours[e[0]], alpha=shade[e[1]], ms=7 if e[2] == 0 else 4, markeredgecolor="white", lw=0.5)
    # One least-squares line per alpha through that alpha's events.
    for a in ALPHAS:
        pick = np.array([e[0] == a for e in events])
        if pick.sum() < 3:
            continue
        sa, ia = np.polyfit(x[pick], y[pick], 1)
        ra = np.corrcoef(x[pick], y[pick])[0, 1]
        xa = np.linspace(x[pick].min(), x[pick].max(), 2)
        ax.plot(xa, sa * xa + ia, color=alpha_colours[a], lw=1.4, ls="--", label=f"alpha {a:g}: slope {sa:+.3f}, r {ra:+.2f}")
    ax.set_xlabel(lx); ax.set_ylabel(ly)
    ax.set_title(f"{ly} against {lx}", loc="left", fontsize=10)
    ax.legend(fontsize=7, loc="best")
    L.tidy_axes(ax)
fig.suptitle(f"Pairs of kernel terms at the generalisation event, {len(events)} runs; dashed lines are the least-squares fit within each alpha "
             "(points coloured by alpha, large for seed 0)")
fig.tight_layout()


## 4. A representative cell

Alpha 1 with no decay: the losses, the test accuracy, and the three kernel terms, with both events marked.


In [ ]:
run = grid[(1.0, 0.0)]
g = L.accuracy_crossings(run, TEST_ACC_LEVEL)  # the two events, for the dots on every panel
fig = L.plot_series([run], ["train_loss", "test_loss", "test_acc"], ["alpha 1, no decay"], [L.BLUE],
                    ["training loss", "test loss", "test accuracy"], log_y=("train_loss", "test_loss"),
                    crossings=[g], log_x=False, mark_train=True,
                    suptitle="One cell, alpha 1 with no decay")
fig = L.plot_series([run], ["S_c", "R_c", "A_t"], ["alpha 1, no decay"], [L.BLUE],
                    ["scale S_t: the kernel norm grows about fivefold", "rotation R_t: the eigenbasis turns by a few percent",
                     "alignment A_t: peaks near the generalisation event"],
                    crossings=[g], log_x=False, mark_train=True,
                    suptitle="The three centred kernel terms")

## 5. Is that ordering of events shared across runs?

Two things stand out in the representative cell: the alignment peaks before the generalisation event, and the event comes just after the fast rise of the scale and rotation curves has flattened. This section checks both over every run with a generalisation event, seed 0 and the extra seeds.

Each panel puts the generalisation step along the bottom and one landmark of the run up the side, on log axes with the diagonal drawn; a point below the diagonal is a run where the landmark came before generalisation. The landmarks are the alignment peak, the step at which the alignment first reaches 95 percent of its peak, and the knees of the scale and rotation curves, taken as the step at which each first reaches 80 percent of its value at the end of the run. The peak itself is a poor landmark under decay, because the alignment keeps creeping up after generalisation and the peak lands at the end of the run; the 95 percent step is the fairer test of whether the alignment had essentially finished rising.


In [ ]:
all_runs = {**{(a, ek, 0): run for (a, ek), run in grid.items()}, **extra}
all_rows = {**{(a, ek, 0): r for (a, ek), r in rows.items()}, **extra_rows}

def knee(run, key, fraction=0.8):
    """First step at which the recorded value reaches the given fraction of its value at the end of the run."""
    h = run["history"]
    return L.first_step_at_least(run, key, fraction * h[key][-1])

records = []
for k, r in all_rows.items():
    if r["t_test"] is None:
        continue
    run = all_runs[k]
    records.append(dict(alpha=k[0], t_test=r["t_test"], peak=L.peak_step(run, "A_t"),
                        kA=L.first_step_at_least(run, "A_t", 0.95 * r["A_peak"]),  # the alignment's own knee
                        A_ratio=r["A_at_test"] / r["A_peak"], kS=knee(run, "S_c"), kR=knee(run, "R_c"),
                        S_frac=r["S_at_test"] / run["history"]["S_c"][-1], R_frac=r["R_at_test"] / run["history"]["R_c"][-1]))

fig, axes = plt.subplots(1, 4, figsize=(19, 4.4))
for ax, key, label in [(axes[0], "peak", "step of the alignment peak"), (axes[1], "kA", "step where alignment reaches 95% of its peak"),
                       (axes[2], "kS", "step where scale reaches 80% of its final value"),
                       (axes[3], "kR", "step where rotation reaches 80% of its final value")]:
    for rec in records:
        ax.plot([rec["t_test"]], [rec[key]], "o", color=alpha_colours[rec["alpha"]], ms=6, alpha=0.8, markeredgecolor="white", lw=0.5)
    lo = min(min(rec["t_test"], rec[key]) for rec in records) * 0.7
    hi = max(max(rec["t_test"], rec[key]) for rec in records) * 1.4
    ax.plot([lo, hi], [lo, hi], color=L.GRAY, lw=1, ls="--")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("generalisation step"); ax.set_ylabel(label)
    below = sum(rec[key] < rec["t_test"] for rec in records)
    ax.set_title(f"{below} of {len(records)} runs below the diagonal", loc="left", fontsize=10)
    L.tidy_axes(ax)
for a in ALPHAS:
    axes[0].plot([], [], "o", color=alpha_colours[a], label=f"alpha {a:g}")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Where the generalisation event falls relative to the alignment peak and to the knees of the scale and rotation curves")
fig.tight_layout()

peak_ratio = np.array([rec["peak"] / rec["t_test"] for rec in records])
A_ratio = np.array([rec["A_ratio"] for rec in records])
for key, name in [("S_frac", "scale"), ("R_frac", "rotation")]:
    v = np.array([rec[key] for rec in records])
    print(f"{name} at the event: median {np.median(v):.0%} of its final value, range {v.min():.0%} to {v.max():.0%}")


The same test at several levels. Each row is one term and each column one level: the landmark is the first step at which the alignment reaches that share of its peak, or the scale and rotation reach that share of their value at the end of the run. The count in each title is the number of runs whose landmark came before the generalisation event.


In [ ]:
LEVELS = [0.5, 0.7, 0.9, 0.95]
terms = [("A_t", "alignment", "peak"), ("S_c", "scale", "final"), ("R_c", "rotation", "final")]
keys_with_event = [k for k, r in all_rows.items() if r["t_test"] is not None]
all_pts = []  # every landmark step drawn, to set one axis range for the whole grid

fig, axes = plt.subplots(len(terms), len(LEVELS), figsize=(4.2 * len(LEVELS), 3.9 * len(terms)), sharex=True, sharey=True)
for row, (key, name, ref) in enumerate(terms):
    for col, level in enumerate(LEVELS):
        ax = axes[row, col]
        pts = []
        for k in keys_with_event:
            run, r = all_runs[k], all_rows[k]
            target = level * (r["A_peak"] if ref == "peak" else run["history"][key][-1])
            landmark = L.first_step_at_least(run, key, target)
            if landmark is not None:
                pts.append((r["t_test"], landmark, k[0]))
        for t, s, a in pts:
            ax.plot([t], [s], "o", color=alpha_colours[a], ms=5, alpha=0.8, markeredgecolor="white", lw=0.5)
        all_pts.extend(s for t, s, a in pts)
        below = sum(s < t for t, s, a in pts)
        ax.set_title(f"{name} at {level:.0%} of {ref}: {below} of {len(pts)} before the event", loc="left", fontsize=9)
        ax.set_xscale("log"); ax.set_yscale("log")
        if row == len(terms) - 1:
            ax.set_xlabel("generalisation step")
        if col == 0:
            ax.set_ylabel("landmark step")
        L.tidy_axes(ax)
lo, hi = min(all_pts) * 0.7, 150000  # same range everywhere so the panels compare directly
for ax in axes.flat:
    ax.plot([lo, hi], [lo, hi], color=L.GRAY, lw=1, ls="--")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
for a in ALPHAS:
    axes[0, 0].plot([], [], "o", color=alpha_colours[a], label=f"alpha {a:g}")
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("Landmarks of each kernel term against the generalisation event, at four levels; below the diagonal means the landmark came first")
fig.tight_layout()


## 6. H1: is the alignment at the crossing shared across cells?

Hypothesis H1 says generalisation begins when the alignment crosses one threshold shared by every cell.
Each panel shows one kernel term read at the generalisation event: every run is a point, grouped by alpha
and coloured by decay, and the band is the mean and standard deviation over cells at seed 0. A flat band
with points inside it is a shared threshold, and the report's kill criterion is a max-to-min ratio above
two. The kernel norm ratio is the contrast, the term that does not carry a threshold.

In [ ]:
def at_event(key, transform=lambda x: x):
    """Value of one kernel term at the generalisation event for every run that has one, keyed by (alpha, decay, seed)."""
    vals = {(a, ek, 0): transform(r[key]) for (a, ek), r in rows.items() if r[key] is not None}
    vals.update({k: transform(r[key]) for k, r in extra_rows.items() if r[key] is not None})
    return vals

panels = [("A_at_test", "alignment A", lambda x: x),
          ("R_at_test", "rotation R", lambda x: x),
          ("S_at_test", "kernel norm ratio exp(S)", np.exp)]
decay_colours = dict(zip(DECAYS, L.RAMP))
offsets = dict(zip(DECAYS, np.linspace(-0.12, 0.12, len(DECAYS))))  # spread the decay levels within an alpha group
summary = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (key, label, transform) in zip(axes, panels):
    vals = at_event(key, transform)
    seed0 = np.array([v for (a, ek, s), v in vals.items() if s == 0])
    for (a, ek, s), v in vals.items():
        ax.plot([np.log2(a) + offsets[ek]], [v], "o", color=decay_colours[ek], ms=7 if s == 0 else 4, alpha=1 if s == 0 else 0.5)
    ax.axhspan(seed0.mean() - seed0.std(ddof=1), seed0.mean() + seed0.std(ddof=1), color=L.GRAY, alpha=0.12)
    ax.axhline(seed0.mean(), color=L.GRAY, lw=1, ls="-.")
    ax.set_xticks([np.log2(a) for a in ALPHAS]); ax.set_xticklabels([f"alpha {a:g}" for a in ALPHAS])
    ratio, cv = seed0.max() / seed0.min(), seed0.std(ddof=1) / seed0.mean()
    ax.set_title(f"{label}: max/min {ratio:.2f}, spread {cv:.0%}", loc="left", fontsize=10)
    if key == "S_at_test":
        ax.set_yscale("log")
    L.tidy_axes(ax)
    summary.append((label, seed0.mean(), seed0.min(), seed0.max(), ratio, cv))
for ek in DECAYS:
    axes[0].plot([], [], "o", color=decay_colours[ek], label=f"eta kappa {ek:g}")
axes[0].plot([], [], "o", color=L.GRAY, ms=4, alpha=0.5, label="other seeds")
axes[0].legend(fontsize=7, loc="lower left", ncol=2)
fig.suptitle("Kernel terms at the generalisation event: alignment, rotation and scale (spread is the standard deviation over cells as a share of the mean)")
fig.tight_layout()

A_star = np.array([v for (a, ek, s), v in at_event("A_at_test").items() if s == 0])  # used by the H2 section
within = [np.std([v for (a2, e2, s), v in at_event("A_at_test").items() if (a2, e2) == (a, ek)], ddof=1)
          for (a, ek) in rows if sum((a2, e2) == (a, ek) for (a2, e2, s) in at_event("A_at_test")) > 1]
print(f"{'term at the event':>38} {'mean':>8} {'min':>8} {'max':>8} {'max/min':>8} {'spread':>7}")
for label, m, lo, hi, ratio, cv in summary:
    print(f"{label:>38} {m:>8.4f} {lo:>8.4f} {hi:>8.4f} {ratio:>8.2f} {cv:>7.1%}")
print(f"\nalignment spread across cells {A_star.std(ddof=1):.4f}, against {np.mean(within):.4f} between seeds of one cell "
      f"(mean over {len(within)} cells with several seeds)")


## 7. Does the threshold carry information?

Section 6 shows that the alignment at the event is nearly the same in every cell. On its own that is not
enough to call it a clock. The alignment only moves from about 0.089 at initialisation to about 0.12, so
it spends most of training inside a narrow band, and a quantity confined to a narrow band takes nearly the
same value whenever it is read. The tightness in section 6 would then say nothing about the event.

The test is a placebo. Read every run at a different run's generalisation step and measure the spread
again. If the threshold is real the spread widens, because the run is now being read at the wrong time. If
the alignment is merely confined, the spread does not change. The permutation is repeated two thousand
times with no run left at its own step, and the reported p is the share of permutations whose spread is no
wider than the spread at the true events.

The first table is a weaker version of the same question, reading every run at other landmarks and at
fixed steps.

In [ ]:
# Every saved run that generalised, with its own generalisation step.
gen = [(run, L.accuracy_crossings(run, TEST_ACC_LEVEL)) for run in {**{k: v for k, v in grid.items()},
                                                                    **{k: v for k, v in extra.items()}}.values()]
gen = [(run, g) for run, g in gen if not g["censored"]]
print(f"{len(gen)} runs generalised.")

def spread(v):
    """The max-to-min ratio and the standard deviation as a share of the mean."""
    v = np.asarray(v, float)
    return v.max() / v.min(), v.std(ddof=1) / v.mean()

TERMS = [("A", "A_t", lambda x: x), ("R", "R_c", lambda x: x), ("exp(S)", "S_c", np.exp)]

landmarks = {
    "generalisation event": lambda run, g: g["t_test"],
    "memorisation event": lambda run, g: g["t_train"],
    "fixed step 20000": lambda run, g: 20000,
    "fixed step 50000": lambda run, g: 50000,
    "end of run": lambda run, g: int(run["history"]["step"][-1]),
}
print(f"\n{'read at':>24}" + "".join(f"{nm + ' max/min':>16}{nm + ' spread':>15}" for nm, _, _ in TERMS))
for nm, at in landmarks.items():
    cols = ""
    for _, key, tr in TERMS:
        q, cv = spread([tr(L.value_at_step(run, key, at(run, g))) for run, g in gen])
        cols += f"{q:>16.2f}{cv:>14.1%} "
    print(f"{nm:>24}{cols}")

In [ ]:
# The placebo: every run read at another run's generalisation step, two thousand times.
rng = np.random.default_rng(0)
steps = np.array([g["t_test"] for _, g in gen])
observed = {nm: spread([tr(L.value_at_step(run, key, g["t_test"])) for run, g in gen])[1]
            for nm, key, tr in TERMS}
draws = {nm: [] for nm, _, _ in TERMS}
for _ in range(2000):
    perm = rng.permutation(len(gen))
    while np.any(perm == np.arange(len(gen))):  # no run may keep its own step
        perm = rng.permutation(len(gen))
    for nm, key, tr in TERMS:
        draws[nm].append(spread([tr(L.value_at_step(gen[i][0], key, steps[perm[i]]))
                                 for i in range(len(gen))])[1])

print(f"{'term':>8} {'spread at its own event':>24} {'median at a swapped event':>26} {'tightening':>11} {'p':>7}")
for nm, _, _ in TERMS:
    d = np.array(draws[nm]); o = observed[nm]
    print(f"{nm:>8} {o:>23.1%} {np.median(d):>25.1%} {np.median(d) / o:>10.1f}x {np.mean(d <= o):>7.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (nm, _, _) in zip(axes, TERMS):
    d = np.array(draws[nm])
    ax.hist(d, bins=40, color=L.RAMP[1], edgecolor="white", lw=0.4)
    ax.axvline(observed[nm], color=L.ORANGE, lw=2)
    ax.set_xlabel(f"spread of {nm} across runs")
    ax.set_ylabel("permutations")
    ax.set_title(f"{nm}: own event {observed[nm]:.1%}, swapped {np.median(d):.1%}", loc="left", fontsize=10)
    L.tidy_axes(ax)
axes[0].annotate("spread at the true events", xy=(observed["A"], 0), xytext=(0.45, 0.8),
                 textcoords="axes fraction", fontsize=8, color=L.ORANGE,
                 arrowprops=dict(arrowstyle="->", color=L.ORANGE, lw=1))
fig.suptitle("Placebo test: the spread of each term when every run is read at another run's generalisation step")
fig.tight_layout()

## 8. H2: does scale matter once alignment is known?

**The question.** H2 says the alignment is the whole clock: once you know how far a run's alignment has
come, the size of its kernel tells you nothing more about when it will generalise. The report specifies
the test as a censored regression of the log grokking time on an alignment-crossing time and a scale
summary, with a kill criterion of a scale term that stays significant with the alignment in the model.

**Version 1 already fails this test.** Its partial correlation of the grokking time with the scale term,
given the alignment-crossing time, is -0.730 with an interval of -0.912 to -0.424, which excludes zero.
The last cell of this section reproduces that number. It is reported here rather than left in a figure
caption, because it is the kill criterion being met.

**Why it is still not the test the report specifies.** Version 1 computes partial correlations on the runs
that generalised and that reach the mean alignment level from section 6. The censored runs are dropped, so
the censoring the report asks to model never enters. And the level is a mean of values that vary, so every
run whose alignment peaked below that mean is dropped as well, including runs that generalised. The first
cell below counts what that costs.

**What this section does instead.** It reports the version 1 statistic for comparison, then runs the
censored regression with `L.censored_log_fit`, the Tobit estimator in the shared module, at a lower
alignment level that the censored runs can reach. It then sweeps the level, because the answer turns out
to depend on it.

In [ ]:
# Every saved run that memorised, censored or not. A censored run has a lower bound on its grokking time.
allr = {**grid, **{k: v for k, v in extra.items()}}
pool = []
for run in allr.values():
    g = L.accuracy_crossings(run, TEST_ACC_LEVEL)
    if g["t_train"] is not None and g["t_grok"] is not None and g["t_grok"] > 0:
        pool.append((run, g))
n_cens = sum(g["censored"] for _, g in pool)
print(f"{len(pool)} runs memorised, of which {n_cens} are censored.")

print(f"\n{'alignment level':>16} {'runs reaching it':>18} {'censored runs reaching it':>26} "
      f"{'generalising runs dropped':>26}")
for level in [0.095, 0.100, 0.105, 0.110, 0.115, float(A_star.mean())]:
    reach = [(run, g) for run, g in pool if L.first_step_at_least(run, "A_t", level) is not None]
    dropped = sum(1 for run, g in pool if not g["censored"]
                  and L.first_step_at_least(run, "A_t", level) is None)
    tag = "  <- the level version 1 used" if abs(level - A_star.mean()) < 1e-9 else ""
    print(f"{level:>16.4f} {len(reach):>18} {sum(g['censored'] for _, g in reach):>26} {dropped:>26}{tag}")

In [ ]:
def design(level, include_censored=True):
    """Grokking times, censoring flags and predictors for every run that reaches an alignment level.

    The predictors are the log of the step at which the alignment first reaches the level and the
    centred scale term read at that same step, so both are defined for censored runs too.
    """
    rows = []
    for run, g in pool:
        if g["censored"] and not include_censored:
            continue
        t_A = L.first_step_at_least(run, "A_t", level)
        if t_A is None:
            continue
        rows.append((g["t_grok"], g["censored"], np.log(t_A), L.value_at_step(run, "S_c", t_A)))
    t, c, tA, S = (np.array(x) for x in zip(*rows))
    return t, c.astype(bool), tA, S

def report(level, cols, names, include_censored=True, n_boot=400):
    """Fit log t_grok on an intercept and the named predictors, with right censoring."""
    t, c, tA, S = design(level, include_censored)
    X = np.c_[np.ones(len(t))]
    for col in cols:
        X = np.c_[X, {"tA": tA, "S": S}[col]]
    f = L.censored_log_fit(X, t, c, n_boot=n_boot, seed=0)
    print(f"  {' + '.join(names)}   n={len(t)}, censored={int(c.sum())}")
    for nm, b, lo, hi in zip(["intercept"] + names, f["coef"], f["lo"], f["hi"]):
        mark = "   significant" if (lo > 0 or hi < 0) else ""
        print(f"    {nm:22} {b:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]{mark}")
    return f

LEVEL = 0.105  # low enough that most censored runs reach it, high enough to be past initialisation
print(f"Censored regression of log grokking time, alignment level {LEVEL}:")
report(LEVEL, ["tA"], ["log alignment time"])
report(LEVEL, ["S"], ["scale at crossing"])
report(LEVEL, ["tA", "S"], ["log alignment time", "scale at crossing"])

In [ ]:
# The same fit at six alignment levels, with and without the censored runs.
print("log t_grok ~ 1 + log alignment time + scale at crossing")
print(f"{'level':>7} {'sample':>20} {'n':>4} {'cens':>5} {'alignment term':>26} {'scale term':>26} {'kill?':>6}")
sweep = []
for level in [0.095, 0.100, 0.105, 0.110, 0.115, float(A_star.mean())]:
    for inc, label in [(True, "all runs"), (False, "survivors only")]:
        t, c, _, _ = design(level, inc)
        if len(t) < 10:
            continue
        f = L.censored_log_fit(np.c_[np.ones(len(t)), design(level, inc)[2], design(level, inc)[3]],
                               t, c, n_boot=400, seed=0)
        (bA, bS), (loA, loS), (hiA, hiS) = f["coef"][1:], f["lo"][1:], f["hi"][1:]
        killed = "yes" if (loS > 0 or hiS < 0) else "no"
        sweep.append((level, label, bS, loS, hiS, killed))
        print(f"{level:>7.4f} {label:>20} {len(t):>4} {int(c.sum()):>5} "
              f"{bA:>+9.3f} [{loA:+.3f},{hiA:+.3f}] {bS:>+9.3f} [{loS:+.3f},{hiS:+.3f}] {killed:>6}")

hit = sum(1 for *_, k in sweep if k == "yes")
print(f"\nThe scale term is significant alongside the alignment in {hit} of {len(sweep)} specifications, "
      "and its sign changes across the range of levels.")
print("H2 fails as version 1 measured it and fails again at several levels here, but the grid does not "
      "fix the size or the sign of the scale term, so the strength of the failure is not established.")

In [ ]:
# Version 1's statistic, reproduced exactly: partial correlations on the runs that generalised and that
# reach the mean alignment level, with the scale summary read at the generalisation event.
def residual(v, z):
    Z = np.c_[np.ones(len(z)), z]
    return v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]

def partial_corr(y, x, z):
    return np.corrcoef(residual(y, z), residual(x, z))[0, 1]

rng = np.random.default_rng(0)
def interval(y, x, z, n=2000):
    b = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if np.std(x[i]) > 0 and np.std(z[i]) > 0:
            b.append(partial_corr(y[i], x[i], z[i]))
    return np.percentile(b, [2.5, 97.5])

A_level = float(A_star.mean())
R_level = float(np.mean([L.value_at_step(run, "R_c", g["t_test"]) for run, g in gen]))
rows_v1 = [(run, g) for run, g in pool
           if not g["censored"] and L.first_step_at_least(run, "A_t", A_level) is not None]
y = np.log([g["t_grok"] for _, g in rows_v1])
t_A = np.log([L.first_step_at_least(run, "A_t", A_level) for run, _ in rows_v1])
S_ev = np.array([L.value_at_step(run, "S_c", g["t_test"]) for run, g in rows_v1])

print(f"Version 1's sample: {len(y)} runs, none censored, alignment level {A_level:.4f}.")
for nm, x, z in [("alignment time given scale", t_A, S_ev), ("scale given alignment time", S_ev, t_A)]:
    lo, hi = interval(y, x, z)
    mark = "   significant" if (lo > 0 or hi < 0) else ""
    print(f"  {nm:30} partial r {partial_corr(y, x, z):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]{mark}")
print("  The scale term excludes zero, so H2's kill criterion is met on version 1's own numbers.")

# The same comparison with the shape term in place of the alignment.
tR = np.array([L.first_step_at_least(run, "R_c", R_level) or np.nan for run, _ in rows_v1], float)
ok = ~np.isnan(tR)
print(f"\nThe shape term in place of the alignment, level {R_level:.4f}, {int(ok.sum())} runs:")
for nm, x, z in [("shape time given scale", np.log(tR[ok]), S_ev[ok]),
                 ("scale given shape time", S_ev[ok], np.log(tR[ok]))]:
    lo, hi = interval(y[ok], x, z)
    mark = "   significant" if (lo > 0 or hi < 0) else ""
    print(f"  {nm:30} partial r {partial_corr(y[ok], x, z):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]{mark}")
print("  The shape term predicts the grokking time at least as well as the alignment does, and the scale "
      "term is no longer significant beside it. That comparison is what bears on H.")

## 9. Each term alone

Section 8 puts the terms in the same model, where they compete. This section asks the simpler question
that the report also needs answering: taken one at a time, which term predicts the grokking time?

Two framings are given because neither is clean on its own. The first uses the step at which each term
first reaches a level, so all three predictors are times. A time predicting a time flatters every
predictor, and the reading to take from that table is the gap between them rather than any single fit.
The second reads each term as a level at a fixed step, which removes that advantage but fixes the moment
of measurement rather than letting each run set its own. Read that table by the residual spread rather
than by significance alone, since on this sample every term is significant at both steps.

The levels for the first table are chosen low enough that all runs reach them, so the censored runs stay
in. Residual sigma is the spread of the log grokking time left unexplained, and the share of variance is
measured against a fit with an intercept and nothing else.

In [ ]:
from scipy.stats import spearmanr

# Levels low enough that every run reaches them, so the censored runs are not selected out.
ALONE = {"alignment time": ("A_t", 0.100), "shape time": ("R_c", 0.050), "scale time": ("S_c", 0.3)}
common = [(run, g) for run, g in pool
          if all(L.first_step_at_least(run, key, lv) is not None for key, lv in ALONE.values())]
t = np.array([g["t_grok"] for _, g in common], float)
c = np.array([g["censored"] for _, g in common])
print(f"{len(common)} runs reach all three levels, of which {int(c.sum())} are censored.")

base = L.censored_log_fit(np.c_[np.ones(len(t))], t, c, n_boot=200, seed=0)
sigma_0 = np.exp(base["log_sigma"])
print(f"With an intercept and nothing else the residual sigma is {sigma_0:.3f}.")

print(f"\n{'single predictor':>18} {'slope':>24} {'residual sigma':>15} {'variance explained':>19} "
      f"{'Spearman':>9}")
alone = {}
for label, (key, lv) in ALONE.items():
    x = np.log([L.first_step_at_least(run, key, lv) for run, _ in common])
    f = L.censored_log_fit(np.c_[np.ones(len(t)), x], t, c, n_boot=400, seed=0)
    sigma = np.exp(f["log_sigma"])
    alone[label] = sigma
    print(f"{label:>18} {f['coef'][1]:>+8.3f} [{f['lo'][1]:+.3f},{f['hi'][1]:+.3f}] {sigma:>15.3f} "
          f"{1 - (sigma / sigma_0) ** 2:>18.1%} {spearmanr(x[~c], np.log(t[~c])).statistic:>9.3f}")
best = min(alone, key=alone.get)
print(f"\nTaken alone the {best.split()[0]} term leaves the smallest residual spread, and both it and "
      "the alignment leave much less than the scale term does.")

In [ ]:
# The same question with each term read as a level at a fixed step, so no predictor is itself a time.
print(f"{'read at':>10} {'term':>16} {'slope':>24} {'residual sigma':>15} {'Spearman':>9}")
for step in [5000, 20000]:
    for label, key, tr in [("alignment A", "A_t", lambda x: x), ("shape R", "R_c", lambda x: x),
                           ("scale exp(S)", "S_c", np.exp)]:
        x = np.array([tr(L.value_at_step(run, key, step)) for run, _ in common])
        f = L.censored_log_fit(np.c_[np.ones(len(t)), x], t, c, n_boot=400, seed=0)
        mark = "  significant" if (f["lo"][1] > 0 or f["hi"][1] < 0) else ""
        print(f"{step:>10} {label:>16} {f['coef'][1]:>+9.3f} [{f['lo'][1]:+.3f},{f['hi'][1]:+.3f}] "
              f"{np.exp(f['log_sigma']):>15.3f} {spearmanr(x[~c], np.log(t[~c])).statistic:>+9.3f}{mark}")
print("\nEarly in training the alignment and the shape term are close and both are far ahead of the "
      "scale term. Later all three are still significant, but the shape term leaves about half the "
      "residual spread the alignment leaves, so the gap between them widens as training goes on.")
print("The shape term is blind to the labels, so a shape term that predicts at least as well as the "
      "alignment is evidence that the timing is set by reshaping rather than by task-relevant "
      "reshaping, which is the part of H that this grid does not support.")